# Rainfall Prediction Classifier: Melbourne, Australia

## Problem
Predict whether it will rain **today** in the Melbourne area, using the full weather record from **yesterday**. This is a binary classification problem (`RainToday`: Yes / No).

## Data
Daily weather observations (2008 to 2017) from Australia's Bureau of Meteorology, via the Kaggle "Rain in Australia" dataset. Fields include temperature, humidity, pressure, wind, cloud cover, and rainfall.

## Approach
1. Clean the data and engineer a `Season` feature
2. Build a preprocessing + model pipeline (scaling, one-hot encoding)
3. Tune a Random Forest with grid search and cross-validation
4. Compare it against Logistic Regression
5. Evaluate with metrics suited to imbalanced classes, not just accuracy

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [2]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/_0eYOqji3unP1tDNKWZMjg/weatherAUS-2.csv"
df = pd.read_csv(url)
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


## Explore missing values
Count the missing values in each column.

**Observation:** `Sunshine`, `Evaporation`, and the `Cloud` columns have a large share of missing values: [fill in the counts]. Imputing that much data would add a lot of guesswork, so I'll drop incomplete rows instead.

In [3]:
print(df.shape)
df.isna().sum().sort_values(ascending=False)

(145460, 23)


Sunshine         69835
Evaporation      62790
Cloud3pm         59358
Cloud9am         55888
Pressure9am      15065
Pressure3pm      15028
WindDir9am       10566
WindGustDir      10326
WindGustSpeed    10263
Humidity3pm       4507
WindDir3pm        4228
Temp3pm           3609
RainTomorrow      3267
Rainfall          3261
RainToday         3261
WindSpeed3pm      3062
Humidity9am       2654
Temp9am           1767
WindSpeed9am      1767
MinTemp           1485
MaxTemp           1261
Location             0
Date                 0
dtype: int64

## Drop rows with missing values
Dropping incomplete rows keeps things simple. [N] rows remain, which is still plenty to train on, so imputation isn't needed.

In [4]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56420 entries, 6049 to 142302
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           56420 non-null  object 
 1   Location       56420 non-null  object 
 2   MinTemp        56420 non-null  float64
 3   MaxTemp        56420 non-null  float64
 4   Rainfall       56420 non-null  float64
 5   Evaporation    56420 non-null  float64
 6   Sunshine       56420 non-null  float64
 7   WindGustDir    56420 non-null  object 
 8   WindGustSpeed  56420 non-null  float64
 9   WindDir9am     56420 non-null  object 
 10  WindDir3pm     56420 non-null  object 
 11  WindSpeed9am   56420 non-null  float64
 12  WindSpeed3pm   56420 non-null  float64
 13  Humidity9am    56420 non-null  float64
 14  Humidity3pm    56420 non-null  float64
 15  Pressure9am    56420 non-null  float64
 16  Pressure3pm    56420 non-null  float64
 17  Cloud9am       56420 non-null  float64
 18  Cloud3p

## Data leakage considerations

If the goal is to predict **tomorrow's** rain, some columns can't be used in practice because they describe the **whole day**, so they aren't known until the day is over. A forecast made during the day wouldn't have them yet.

Features that rely on the full day:

- **`Rainfall`**: total rain for the day (24 hours), so it's only final at the end of the day.
- **`Sunshine`**: total hours of bright sunshine across the day.
- **`Evaporation`**: measured over a 24-hour period.
- **`MaxTemp` / `MinTemp`**: the day's extremes can only be confirmed once the day has finished.
- **`WindGustSpeed` / `WindGustDir`**: the strongest gust over the whole day.
- **`RainToday`**: depends on whether at least 1 mm fell over the full day.

Features like `Humidity9am` or `Pressure9am` are measured at a fixed time, so they're available earlier in the day and are less of a problem. The 3pm readings are only usable if the prediction is made after 3pm.

**Reframing the problem:** instead of predicting tomorrow from today, I predict *today's* rain using the full weather record from *yesterday*. Then every column is known at prediction time, so all features can be used. I rename `RainToday` → `RainYesterday` and `RainTomorrow` → `RainToday` to match. The data itself doesn't change. Only the question being asked does.

## Reframe the target
Rename the columns to match the reframed question. `RainToday` becomes `RainYesterday` (a feature) and `RainTomorrow` becomes `RainToday` (the target). The data is unchanged, and only the question being asked is different.

In [5]:
df = df.rename(columns={'RainToday': 'RainYesterday',
                        'RainTomorrow': 'RainToday'})

## Narrow to the Melbourne area
Rainfall patterns vary a lot across Australia, so a single model for the whole country would need to be much more complex. I keep three nearby locations (Melbourne, Melbourne Airport, and Watsonia, all within about 18 km of each other) and keep `Location` as a categorical feature to capture small local differences.

This leaves 7,557 rows, which should be enough for a reasonable model.

In [6]:
df = df[df.Location.isin(['Melbourne', 'MelbourneAirport', 'Watsonia'])]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7557 entries, 64191 to 80997
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           7557 non-null   object 
 1   Location       7557 non-null   object 
 2   MinTemp        7557 non-null   float64
 3   MaxTemp        7557 non-null   float64
 4   Rainfall       7557 non-null   float64
 5   Evaporation    7557 non-null   float64
 6   Sunshine       7557 non-null   float64
 7   WindGustDir    7557 non-null   object 
 8   WindGustSpeed  7557 non-null   float64
 9   WindDir9am     7557 non-null   object 
 10  WindDir3pm     7557 non-null   object 
 11  WindSpeed9am   7557 non-null   float64
 12  WindSpeed3pm   7557 non-null   float64
 13  Humidity9am    7557 non-null   float64
 14  Humidity3pm    7557 non-null   float64
 15  Pressure9am    7557 non-null   float64
 16  Pressure3pm    7557 non-null   float64
 17  Cloud9am       7557 non-null   float64
 18  Cloud3pm

## Feature engineering: Season
Weather is seasonal, so I convert `Date` into a `Season` feature and drop the raw date. Seasons follow the southern hemisphere (December to February is summer). I leave `Year` out for now.

In [7]:
def date_to_season(date):
    month = date.month
    if month in (12, 1, 2):
        return 'Summer'
    elif month in (3, 4, 5):
        return 'Autumn'
    elif month in (6, 7, 8):
        return 'Winter'
    else:
        return 'Spring'

df['Date'] = pd.to_datetime(df['Date'])
df['Season'] = df['Date'].apply(date_to_season)
df = df.drop(columns='Date')
df.head()

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainYesterday,RainToday,Season
64191,MelbourneAirport,11.2,19.9,0.0,5.6,8.8,SW,69.0,W,SW,...,37.0,1005.1,1006.4,7.0,7.0,15.9,18.1,No,Yes,Summer
64192,MelbourneAirport,7.8,17.8,1.2,7.2,12.9,SSE,56.0,SW,SSE,...,43.0,1018.0,1019.3,6.0,7.0,12.5,15.8,Yes,No,Summer
64193,MelbourneAirport,6.3,21.1,0.0,6.2,10.5,SSE,31.0,E,S,...,35.0,1020.8,1017.6,1.0,7.0,13.4,19.6,No,No,Summer
64194,MelbourneAirport,8.1,29.2,0.0,6.4,12.5,SSE,35.0,NE,SSE,...,23.0,1016.2,1012.8,5.0,4.0,16.0,28.2,No,No,Summer
64195,MelbourneAirport,9.7,29.0,0.0,7.4,12.3,SE,33.0,SW,SSE,...,31.0,1011.9,1010.3,6.0,2.0,19.4,27.1,No,No,Summer


## Define features and target
`X` holds all the predictors and `y` is `RainToday`. I check the class balance first, since it determines how the model should be evaluated.

In [8]:
X = df.drop(columns='RainToday')
y = df['RainToday']

print(y.value_counts())
print(y.value_counts(normalize=True))

RainToday
No     5766
Yes    1791
Name: count, dtype: int64
RainToday
No     0.763001
Yes    0.236999
Name: proportion, dtype: float64


### What the class counts show
- **Rain frequency:** it rains on about 23.7% of days in this area.
- **Baseline:** always predicting "No rain" would be right about 76.3% of the time, so any useful model must beat that.
- **Balance:** the dataset is **imbalanced**, with far more dry days than rainy ones.
- **Next steps:** use a stratified train/test split and cross-validation to keep the class ratio consistent, and judge the model with precision, recall, and F1 for the rain class, not accuracy alone.

## Train/test split
Hold out 20% of the data for testing. I use `stratify=y` so both sets keep the same rain/no-rain ratio (about 24% rain). Without it, a random split could skew the class balance, and with imbalanced data that makes evaluation unreliable.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Preprocessing
Split the columns into numeric and categorical features, since each needs different handling:
- **Numeric:** scaled with `StandardScaler`. Random Forests don't need scaling, but Logistic Regression does, and I'm reusing the same preprocessing for both.
- **Categorical:** one-hot encoded. `handle_unknown='ignore'` stops the model from crashing if the test set contains a category it didn't see in training.

Both transformers go inside the pipeline, so during cross-validation they are fit only on the training folds. This avoids data leakage.

In [10]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(exclude='number').columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


Numeric: ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
Categorical: ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainYesterday', 'Season']


## Model 1: Random Forest with grid search
Combine the preprocessor and a Random Forest into a single pipeline, then use `GridSearchCV` to search over tree count, depth, and minimum samples per split. I use stratified 5-fold cross-validation so every fold has the same class balance.

In [11]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', verbose=2)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=50; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=50; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=50; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=50; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=50; total time=   0.2s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=100; total time=   0.3s
[CV] END classifier__max_depth=None, classifier__min_samples_split=2, classifier__n_estimators=100; total time=

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['MinTemp',
                                                                          'MaxTemp',
                                                                          'Rainfall',
                                                                          'Evaporation',
                                                                          'Sunshine',
                                                                          'WindGustSpeed',
                                                                          'WindSpeed9am',
                                                                          'WindSpeed3pm',
                                                                          'Humidity9am',
                                                                          'Humidity3pm',
                                                                          'Pressure9am',
                                                                          'Pres...
                                                                         Pipeline(steps=[('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Location',
                                                                          'WindGustDir',
                                                                          'WindDir9am',
                                                                          'WindDir3pm',
                                                                          'RainYesterday',
                                                                          'Season'])])),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__min_samples_split': [2, 5],
                         'classifier__n_estimators': [50, 100]},
             scoring='accuracy', verbose=2)

In [12]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

test_score = grid_search.score(X_test, y_test)
print("Test set score: {:.2f}".format(test_score))

Best parameters: {'classifier__max_depth': 20, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Best cross-validation score: 0.85
Test set score: 0.84


### First results
The best parameters were `max_depth=20`, `min_samples_split=2`, and `n_estimators=100`. Cross-validation accuracy was 0.85 and test accuracy was 0.84. The two are close, so the model is not badly overfit.

Compared with the 76.3% baseline of always predicting "No rain", this is a modest improvement of about 8 points. But accuracy hides how well the model finds rainy days, which is the harder and more useful job. I look at the classification report and confusion matrix next.

Note: the best values sit at the edge of the grid (the largest depth and tree count searched), so a wider search might improve things slightly.